In [1]:
#| default_exp rest

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [3]:
#| export
from rest.core import init_instance, process_seq
singleton, model_path = init_instance()

In [4]:
model_path = 'pelevin'

In [5]:
#| export
seq_length = 1024

model_path = f'./models/large/{model_path}'
from transformers import GPT2LMHeadModel,GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
model = GPT2LMHeadModel.from_pretrained(model_path).half()
model.cuda()
model.eval();

In [6]:
sum(p.numel() for p in model.parameters())

774030080

In [7]:
#| export
def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
    encoded_prompt = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt").cuda()
    encoded_prompt = encoded_prompt[:,length-(seq_length-1):]
    bad_words_ids = [tokenizer.encode('[')[0], tokenizer.encode('(')[0], tokenizer.encode('1\xa01')[1]]
    linebreak = tokenizer.encode("1\n1")[1]
    lb2 = tokenizer.encode("1 \n")[1]
    bad_words_ids += [] if allow_linebreak else [linebreak, lb2]
    bad_words_ids = [[b] for b in bad_words_ids] + [[linebreak,linebreak]]
    output_sequences = model.generate(
            input_ids=encoded_prompt,
            max_length=length + len(encoded_prompt[0]),
            temperature=1,
            top_k=0,
            top_p=0.9,
            do_sample=True,num_return_sequences=num_samples,
            bad_words_ids = bad_words_ids,
        )
    
    if len(output_sequences.shape) > 2:
            output_sequences.squeeze_()
    generated_sequences = []
    for generated_sequence_idx, generated_sequence in enumerate(output_sequences):
        generated_sequence = generated_sequence.tolist()
        text = tokenizer.decode(generated_sequence, clean_up_tokenization_spaces=True)
        total_sequence = text[len(tokenizer.decode(encoded_prompt[0], clean_up_tokenization_spaces=True)) :]
        generated_sequences.append(total_sequence)

    return process_seq(generated_sequences)

In [9]:
%%time
get_sample(' - ты кто?', 50, 4, False)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


not setting adaptive thresholding
CPU times: user 12.4 s, sys: 488 ms, total: 12.9 s
Wall time: 1.23 s


[' Что-то знакомое… Тебя как зовут? Винни? А я Электра. У нас знаешь как бывает? Приходит дядя, живет в маленькой комнатке, а ты все время под дверью… А тебя где прячут? В холодильнике?',
 ' Не смотри так. Я-тоже. Я демон-но ты мне нравишься. Пожалуйста, не гони. Скажи, куда идти. Пойдем вместе, есть способ. У нас работа одинаковая. Слушай, ты хоть кого-нибудь жалеешь?',
 ' - спросил я, надеясь, что в этот раз он сможет сделать правильные выводы. Похоже, он не стал утруждать себя работой мысли. Вместо этого он ткнул длинным пальцем в меня и прошептал: "Дас ист фантастиш!',
 ' Откуда ты? - спросила она, глядя куда-то в сторону и одновременно не отводя взгляда от сивой головы над штакетником. - Ты, часом, не в кавалерии служил?']